# Notebook 2: QSARmil pipeline configuration

In contrast to the previous notebook, here we consider each step of the modelling pipeline individually and in greater detail. This notebook can serve as a starting point for customizing the workflow, exploring different configurations, and performing more extensive benchmarking across alternative descriptors and multi-instance learning methods.

In [1]:
import numpy as np
import pandas as pd
from rdkit import Chem
from sklearn.metrics import r2_score

### 1. Data load

As an example, we use a publicly available and easily accessible collection of molecular bioactivity datasets introduced in the paper by:

> Van Tilborg, Derek, Alisa Alenicheva, and Francesca Grisoni. "Exposing the limitations of molecular machine learning with activity cliffs." Journal of chemical information and modeling 62.23 (2022): 5938-5951.

From this collection, we select one dataset for demonstration purposes.

In [2]:
url = "https://raw.githubusercontent.com/molML/MoleculeACE/main/MoleculeACE/Data/benchmark_data/CHEMBL2034_Ki.csv"
df_ace = pd.read_csv(url)

df_train = df_ace[df_ace["split"] == "train"][["smiles", "y"]]
df_test = df_ace[df_ace["split"] == "test"][["smiles", "y"]]

### 2. Data validation

The data validation step is straightforward but includes an additional requirement compared to standard 2D modelling pipelines. In most QSAR workflows, input structures are primarily validated by checking whether their SMILES strings are valid and can be successfully parsed into molecular objects.

In contrast, **QSARmil** introduces an extra validation step: verifying whether a valid 3D structure can be generated for each molecule. Molecules that fail this step cannot be used in the 3D modelling pipeline. As a result, the filtering (removal) rate in QSARmil may be higher than in typical 2D QSAR workflows.

In [3]:
from qsarmil.data.input_data import DataValidator

In [4]:
dvalid = DataValidator(num_cpu=30, verbose=True)

df_train = dvalid.filter_dataframe(df_train)
df_test = dvalid.filter_dataframe(df_test)

No rows removed. All molecules are valid.
No rows removed. All molecules are valid.


### 3. Conformer generation

Conformers are generated using the **RDKit** conformer generation module, which applies ETKDG geometry embedding followed by UFF optimization and conformer cleanup. Currently, the generation process can be controlled using the ``num_conf`` parameter, which sets the maximum number of conformers, and an energy gap threshold ``e_tresh`` relative to the most stable conformer; all conformers exceeding this energy window are discarded.

Importantly, the conformer generator accepts **RDKit** molecule objects as input but returns a QSARmil-specific data structure: ``ConformerEnsemble``. This object represents a collection of conformers and provides dedicated methods for multi-instance learning workflows. For example, it allows straightforward access to properties such as the average conformer energy.

This design reflects a core concept in QSARmil, where molecules are represented as structured ensembles on instances. In line with this idea, additional ensemble types are currently under development, including ``FragmentEnsemble``, ``TautomerEnsemble``, and ``MixtureEnsemble``.

In [5]:
from qsarmil.conformer import RDKitConformerGenerator

In [6]:
conf_gen = RDKitConformerGenerator(num_conf=10, e_thresh=50, num_cpu=40, verbose=True)

In [7]:
mols_train = [Chem.MolFromSmiles(smi) for smi in df_train["smiles"]]
mols_test = [Chem.MolFromSmiles(smi) for smi in df_test["smiles"]]

confs_train = conf_gen.run(mols_train)
confs_test = conf_gen.run(mols_test)

Generating conformers: 598/598
Generating conformers: 152/152


In [8]:
# conformer ensemble
ce = confs_train[0]
type(ce)

qsarmil.utils.ensemble.ConformerEnsemble

### 4. Descriptor calculation

Once conformers are generated for each molecule, the next step is to compute **3D molecular descriptors**.
Since each molecule can have multiple conformers, the resulting data are structured as **bags of descriptor vectors**, where each **bag** corresponds to a molecule and each **instance** within the bag corresponds to a conformer.

To manage this process, a **descriptor wrapper** is used.  This wrapper provides a unified interface for applying multiple descriptor calculators sourced from external packages (e.g., **RDKit** and **MolFeat**). It automatically handles descriptor computation for all conformers in each molecule.

In this example, several descriptor types are combined:
- **RDKit-based 3D descriptors:** `RDKitGEOM`, `RDKitAUTOCORR`, `RDKitRDF`, `RDKitMORSE`, `RDKitWHIM`, `RDKitGETAWAY`
- **MolFeat descriptors:** `Pharmacophore3D`, `USRDescriptors`, `ElectroShapeDescriptors`

After computing the raw descriptors, the values are **scaled** using `BagMinMaxScaler` from the `milearn.preprocessing` module.  
This scaling step ensures that descriptor values across conformers and molecules are brought to a consistent range, which helps stabilize and improve model training.

In [9]:
from molfeat.calc import Pharmacophore3D, USRDescriptors, ElectroShapeDescriptors

from qsarmil.descriptor.rdkit import RDKitGEOM, RDKitAUTOCORR, RDKitRDF, RDKitMORSE, RDKitWHIM, RDKitGETAWAY
from qsarmil.descriptor.wrapper import DescriptorWrapper

from milearn.preprocessing import BagMinMaxScaler

In [10]:
desc_calc = DescriptorWrapper(Pharmacophore3D(factory="pmapper"), verbose=True)

In [11]:
x_train = desc_calc.run(confs_train)
x_test = desc_calc.run(confs_test)

Calculating descriptors: 598/598
Calculating descriptors: 152/152


In [12]:
scaler = BagMinMaxScaler()

scaler.fit(x_train)

# descriptors
x_train_scaled = scaler.transform(x_train)
x_test_scaled = scaler.transform(x_test)

# property
y_train, y_test = df_train["y"], df_test["y"]

### 5. Multi-instance method

In QSARmil, there are multiple families of MIL models:

- **Wrapper MIL Networks**  
  These models wrap standard MLPs with MIL pooling strategies.  
  - `BagWrapperMLPNetwork` – pooling applied to bag embeddings  
  - `InstanceWrapperMLPNetwork` – pooling applied to instance embeddings  

- **Classic MIL Networks**  
  Standard architectures where pooling is directly integrated into the MIL framework.  
  - `BagNetwork`  
  - `InstanceNetwork`

- **Attention-Based MIL Networks**  
  Models that learn to assign importance weights to individual conformers.  
  - `AdditiveAttentionNetwork`  
  - `SelfAttentionNetwork`  
  - `HopfieldAttentionNetwork`

- **Other MIL Architectures**  
  - `DynamicPoolingNetwork` – adaptive pooling mechanism that adjusts to bag composition

In [13]:
# MIL wrappers
from milearn.network.regressor import BagWrapperMLPNetworkRegressor, InstanceWrapperMLPNetworkRegressor
from milearn.network.classifier import BagWrapperMLPNetworkClassifier, InstanceWrapperMLPNetworkClassifier

# MIL networks
from milearn.network.regressor import (InstanceNetworkRegressor,
                                       BagNetworkRegressor,
                                       AdditiveAttentionNetworkRegressor,
                                       SelfAttentionNetworkRegressor,
                                       HopfieldAttentionNetworkRegressor,
                                       DynamicPoolingNetworkRegressor)

### 6. Stepwise hyperparameter pptimization

The *milearn* framework also supports **stepwise hyperparameter optimization**.  
This optimizer explores a predefined hyperparameter grid in a **parameter-by-parameter** manner, identifying the best value for each while keeping previously optimized parameters fixed.
 
To run hyperparameter optimization before training, simply call:
```python
model.hopt(x_train_scaled, y_train, param_grid=DEFAULT_PARAM_GRID, verbose=True)

In [14]:
from milearn.network.module.hopt import DEFAULT_PARAM_GRID

In [15]:
# default hopt grid
DEFAULT_PARAM_GRID

{'max_epochs': 1000,
 'early_stopping': True,
 'accelerator': 'cpu',
 'random_seed': 42,
 'verbose': False,
 'activation': ['relu', 'leakyrelu', 'gelu', 'elu', 'silu'],
 'learning_rate': [0.0001, 0.001],
 'batch_size': [32, 512, 1024],
 'weight_decay': [0.0, 1e-05, 0.0001, 0.001, 0.01],
 'tau': [0.01, 0.5, 1.0],
 'instance_dropout': [0.0, 0.2, 0.4, 0.6, 0.8],
 'pool': ['mean', 'sum', 'max', 'lse'],
 'hidden_layer_sizes': [(2048, 1024, 512, 256, 128, 64),
  (256, 128, 64),
  (128,)]}

In [16]:
model = AdditiveAttentionNetworkRegressor()
model.hopt(x_train_scaled, y_train, param_grid=DEFAULT_PARAM_GRID, verbose=True)
model.fit(x_train_scaled, y_train)

Optimizing hyperparameter: activation (5 options)
[1/26 |  3.8% |  0.2 min] Value: relu, Epochs: 35, Loss: 0.6030
[2/26 |  7.7% |  0.2 min] Value: leakyrelu, Epochs: 40, Loss: 0.5771
[3/26 | 11.5% |  0.1 min] Value: gelu, Epochs: 14, Loss: 0.6874
[4/26 | 15.4% |  0.2 min] Value: elu, Epochs: 33, Loss: 0.6513
[5/26 | 19.2% |  0.2 min] Value: silu, Epochs: 38, Loss: 0.6359
Best activation = leakyrelu, val_loss = 0.5771
Optimizing hyperparameter: learning_rate (2 options)
[6/26 | 23.1% |  0.1 min] Value: 0.0001, Epochs: 24, Loss: 0.6766
[7/26 | 26.9% |  0.1 min] Value: 0.001, Epochs: 17, Loss: 0.6217
Best learning_rate = 0.001, val_loss = 0.6217
Optimizing hyperparameter: batch_size (3 options)
[8/26 | 30.8% |  0.1 min] Value: 32, Epochs: 17, Loss: 0.5989
[9/26 | 34.6% |  0.1 min] Value: 512, Epochs: 30, Loss: 0.6595
[10/26 | 38.5% |  0.1 min] Value: 1024, Epochs: 30, Loss: 0.6597
Best batch_size = 32, val_loss = 0.5989
Optimizing hyperparameter: weight_decay (5 options)
[11/26 | 42.3% | 

AdditiveAttentionNetworkRegressor(
  (instance_transformer): Sequential(
    (0): Linear(in_features=2048, out_features=128, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
  )
  (bag_estimator): Linear(in_features=128, out_features=1, bias=True)
  (attention): Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): Tanh()
    (2): Linear(in_features=128, out_features=1, bias=True)
  )
)

### 7. Model prediction

The final step is to use a trained model to generate predictions for new molecules. In this stage, additional functionalities specific to multi-instance learning are available. In particular, some MIL methods can predict not only the overall molecular property but also instance-level (i.e., conformer-level) weights. This second capability is discussed in more detail in the next tutorial.

In [17]:
y_pred = model.predict(x_test_scaled)
w_pred = model.get_instance_weights(x_test_scaled)

In [18]:
r2_score(y_test, y_pred)

0.5150988149567628